In [ ]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/atta00/cpt-hcpcs-codes/" + splits["train"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
df.head()


,index,code,title,category,description,type
0,0,100,"""Anesthesia for Procedures on the Head.Anesthe...",Anesthesiology,"""The provider performs anesthesia services for...",CPT
1,1,102,"""Anesthesia for Procedures on the Head.Anesthe...",Anesthesiology,"""The provider performs anesthesia services for...",CPT
2,2,103,"""Anesthesia for Procedures on the Head.Anesthe...",Anesthesiology,"""The provider performs anesthesia services for...",CPT
3,3,104,"""Anesthesia for Procedures on the Head.Anesthe...",Anesthesiology,"""The provider performs anesthesia services for...",CPT
4,4,120,"""Anesthesia for Procedures on the Head.Anesthe...",Anesthesiology,"""The provider performs anesthesia services for...",CPT


In [ ]:
print("rows:", len(df))
print("cols:", df.columns.tolist())
df.sample(5, random_state=0)

rows: 16056
cols: ['index', 'code', 'title', 'category', 'description', 'type']


,index,code,title,category,description,type
6839,6839,80185,"""Therapeutic Drug Assays.Laboratory. Chemistry...",Pathology,"""The lab analyst performs the technical lab te...",CPT
3460,3460,38220,"""Bone Marrow or Stem Cell Services/Procedures.""",Surgery,"""The provider inserts a long large–bore needle...",CPT
7348,7348,82415,"""Chemistry Procedures.""",Pathology,"""The lab analyst determines the drug level of ...",CPT
15730,15730,S9992,"""Various Services Fees and Costs .""",Mental Health and Substance Abuse,"""Transportation costs to and from trial locati...",HCPS
1352,1352,25101,"""Excision Procedures on the Forearm and Wrist....",Surgery,"""In this procedure the provider incises the wr...",CPT


In [ ]:
import re

def find_col_by_regex(cols, patterns):
    for p in patterns:
        rx = re.compile(p, re.IGNORECASE)
        for c in cols:
            if rx.search(c):
                return c
    return None

code_col = find_col_by_regex(df.columns, [r"^code$", r"hcpcs", r"cpt", r"procedure.*code", r"^cpt_code$", r"^hcpcs_code$"])
desc_col = find_col_by_regex(df.columns, [r"desc", r"description", r"long.*desc", r"short.*desc", r"^text$"])

print("Guessed code_col:", code_col)
print("Guessed desc_col:", desc_col)

assert code_col is not None, "Couldn't find a code column; inspect df.columns"
assert desc_col is not None, "Couldn't find a description/text column; inspect df.columns"


Guessed code_col: code
Guessed desc_col: description


In [ ]:


codes = df[code_col].astype(str).str.strip()

cpt_rx = re.compile(r"^\d{5}$")
hcpcs_rx = re.compile(r"^[A-Z]\d{4}$")

is_cpt = codes.str.match(cpt_rx)
is_hcpcs = codes.str.match(hcpcs_rx)

fmt_summary = pd.DataFrame({
    "count": [is_cpt.sum(), is_hcpcs.sum(), (~(is_cpt | is_hcpcs)).sum()],
    "pct":   [is_cpt.mean(), is_hcpcs.mean(), (~(is_cpt | is_hcpcs)).mean()],
}, index=["CPT_5_digit", "HCPCS_L2", "Other/Unexpected"])

display(fmt_summary)

# Show examples of unexpected codes
unexpected = df.loc[~(is_cpt | is_hcpcs), [code_col, desc_col]].head(20)
display(unexpected)


,count,pct
CPT_5_digit,9381,0.584268
HCPCS_L2,6397,0.398418
Other/Unexpected,278,0.017314


,code,description
0,100,"""The provider performs anesthesia services for..."
1,102,"""The provider performs anesthesia services for..."
2,103,"""The provider performs anesthesia services for..."
3,104,"""The provider performs anesthesia services for..."
4,120,"""The provider performs anesthesia services for..."
5,124,"""The provider performs anesthesia services for..."
6,126,"""The provider performs anesthesia services for..."
7,140,"""The provider performs anesthesia services for..."
8,142,"""The provider performs anesthesia services for..."
9,144,"""The provider performs anesthesia services for..."


In [ ]:
desc = df[desc_col].astype(str)

missing_code = df[code_col].isna().mean()
missing_desc = df[desc_col].isna().mean()

empty_code = (df[code_col].astype(str).str.strip() == "").mean()
empty_desc = (df[desc_col].astype(str).str.strip() == "").mean()

print("Missing code %:", missing_code)
print("Missing desc %:", missing_desc)
print("Empty code %:", empty_code)
print("Empty desc %:", empty_desc)

assert missing_code < 0.001, "Too many missing codes"
assert empty_desc < 0.01, "Too many empty descriptions"


Missing code %: 0.0
Missing desc %: 0.0
Empty code %: 0.0
Empty desc %: 0.0


In [ ]:
n_total = len(df)
n_unique_codes = df[code_col].nunique(dropna=True)
dup_rows = df.duplicated(subset=[code_col]).sum()

print("Total rows:", n_total)
print("Unique codes:", n_unique_codes)
print("Duplicate code rows:", dup_rows)
print("Dup rate:", dup_rows / n_total)

# Inspect top duplicated codes
dup_counts = df[code_col].value_counts().head(20)
display(dup_counts)

# Show duplicated examples
dups = df[df.duplicated(subset=[code_col], keep=False)].sort_values(code_col).head(50)
display(dups[[code_col, desc_col]])


Total rows: 16056
Unique codes: 16056
Duplicate code rows: 0
Dup rate: 0.0


,count
code,
U0002,1
100,1
102,1
103,1
104,1
120,1
124,1
126,1
V5281,1


,code,description


In [ ]:
desc_clean = df[desc_col].astype(str).str.strip()
lens = desc_clean.str.len()

print("Description length stats:")
display(lens.describe(percentiles=[0.01,0.05,0.10,0.25,0.5,0.75,0.9,0.95,0.99]))

# How many are "too short"?
too_short = (lens < 50).mean()
print("Percent descriptions < 10 chars:", too_short)

# Look at examples of very short descriptions
display(df.loc[lens < 15, [code_col, desc_col]].head(30))


Description length stats:


,description
count,16056.000000
mean,182.028089
std,131.333476
min,8.000000
1%,24.000000
5%,34.000000
10%,45.000000
25%,84.000000
50%,158.000000
75%,249.000000


Percent descriptions < 10 chars: 0.1213876432486298


,code,description
9930,A4565,"""Slings"""
9932,A4570,"""Splint"""
9951,A4617,"""Mouth piece"""
9953,A4619,"""Face tent"""
10034,A5055,"""Stoma cap"""
10551,C1769,"""Guide wire"""
10562,C1782,"""Morcellator"""
10949,E0273,"""Bed board"""
11000,E0459,"""Chest wrap"""
11117,E0776,"""IV pole"""


In [ ]:
def search_desc(term, n=10):
    mask = df[desc_col].astype(str).str.contains(term, case=False, regex=False, na=False)
    return df.loc[mask, [code_col, desc_col]].head(n)

display(search_desc("mri", n=15))
display(search_desc("physical therapy", n=15))
display(search_desc("cholesterol", n=15))
display(search_desc("knee", n=15))


,code,description
739,20660,"""Cranial tongs calipers or stereotactic frames..."
1156,23350,"""In this procedure the provider performs an in..."
1823,27369,"""procedure of performing an injection of contr..."
2527,31627,"""In this add–on procedure the provider uses pr..."
4707,53250,"""The bulbourethral glands are a pair of pea–si..."
5359,61736,"""The provider performs laser interstitial ther..."
5360,61737,"""The provider performs laser interstitial ther..."
5361,61750,"""In this procedure the provider removes a port..."
5362,61751,"""In this procedure the provider removes a port..."
5364,61770,"""In this procedure the provider removes a port..."


,code,description
9087,94667,"""The provider performs chest physical therapy ..."
9088,94668,"""The provider performs chest physical therapy ..."
9371,97039,"""Use to report procedures in which the provid..."
9372,97110,"""Therapeutic exercise is a form of physical th..."
9380,97140,"""Manual therapy is a form of physical therapy ..."
9390,97161,"""The provider typically a physical therapist c..."
9391,97162,"""The provider typically a physical therapist c..."
9392,97163,"""The provider typically a physical therapist c..."
9393,97164,"""The provider typically a physical therapist c..."
11588,G0159,"""Services performed by a qualified physical th..."


,code,description
6671,78075,"""The provider performs a functional imaging te..."
6805,80061,"""The lab analyst performs a lipid panel test i..."
7313,82239,"""The lab analyst performs an analysis primaril..."
7353,82465,"""The lab analyst measures the level of total c..."
7357,82495,"""The lab analyst measures the level of chromiu..."
7514,83718,"""The lab analyst performs a test to directly m..."
7515,83719,"""The lab analyst performs a test to directly m..."
7516,83721,"""The lab analyst performs a test to directly m..."
7517,83722,"""The lab analyst performs a test to directly m..."
12603,G9782,"""History of or active diagnosis of familial hy..."


,code,description
163,1220,"""The provider performs anesthesia services for..."
164,1230,"""The provider performs anesthesia services for..."
165,1232,"""The provider performs anesthesia services for..."
166,1234,"""The provider performs anesthesia services for..."
172,1320,"""The provider performs anesthesia services for..."
173,1340,"""The provider performs anesthesia services for..."
174,1360,"""The provider performs anesthesia services for..."
175,1380,"""The provider performs anesthesia services for..."
176,1382,"""The provider performs anesthesia services for..."
179,1400,"""The provider performs anesthesia services for..."


In [ ]:
import os

os.listdir()


['.config', 'sample_data']

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving rates_data.xlsx to rates_data.xlsx


In [ ]:


rate_df = pd.read_excel("rates_data.xlsx", skiprows=1)


rate_df.head()


,Proc Type,Proc Code,Procedure Description,Unit Value,Basic Rate,Child Rate,ER Rate,Conv Ind,ER Ind,Cutback Ind,Prof %,Rental Rate,Non-Physn Med Prac Ind
0,M,0001U,RBC DNA HEA 35 AG 11 BLD GRP,576.00,$576.00,---,--,9,0,0,0.0,$0.00,N
1,M,0003U,ONC OVAR 5 PRTN SER ALG SCOR,760.00,$760.00,---,--,9,0,0,0.0,$0.00,N
2,M,0007U,RX TEST PRSMV UR W/DEF CONF,91.54,$91.54,---,--,9,0,0,0.0,$0.00,N
3,J,00100,ANES PX SALIVARY GLAND W/BX,4.52,$63.33,---,--,2,0,0,0.0,$0.00,Y
4,J,00102,ANES PX PLSTC RPR CLEFT LIP,5.42,$75.93,---,--,2,0,0,0.0,$0.00,Y


In [ ]:
rate_df.columns = (
    rate_df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
)

rate_df.columns.tolist()



['proc_type',
 'proc_code',
 'procedure_description',
 'unit_value',
 'basic_rate',
 'child_rate',
 'er_rate',
 'conv_ind',
 'er_ind',
 'cutback_ind',
 'prof_%',
 'rental_rate',
 'non_physn_med_prac_ind']

In [ ]:
import re

codes = rate_df["proc_code"].astype(str).str.strip().str.upper()

cpt_rx = re.compile(r"^\d{5}$")
hcpcs_rx = re.compile(r"^[A-Z]\d{4}$")

rate_df["is_cpt"] = codes.str.match(cpt_rx)
rate_df["is_hcpcs"] = codes.str.match(hcpcs_rx)

summary = {
    "total_rows": len(rate_df),
    "cpt_%": rate_df["is_cpt"].mean(),
    "hcpcs_%": rate_df["is_hcpcs"].mean(),
    "unexpected_%": (~(rate_df["is_cpt"] | rate_df["is_hcpcs"])).mean()
}

summary


{'total_rows': 19401,
 'cpt_%': np.float64(0.7501159734034328),
 'hcpcs_%': np.float64(0.2325653316839338),
 'unexpected_%': np.float64(0.01731869491263337)}

In [ ]:
def clean_money(col):
    return (
        col.astype(str)
           .str.replace("$", "", regex=False)
           .str.replace(",", "", regex=False)
           .replace({"---": None, "--": None})
           .astype(float)
    )

rate_df["basic_rate_clean"] = clean_money(rate_df["basic_rate"])
rate_df["child_rate_clean"] = clean_money(rate_df["child_rate"])
rate_df["er_rate_clean"] = clean_money(rate_df["er_rate"])
rate_df["rental_rate_clean"] = clean_money(rate_df["rental_rate"])

rate_df[[
    "basic_rate", "basic_rate_clean",
    "child_rate", "child_rate_clean"
]].head()


,basic_rate,basic_rate_clean,child_rate,child_rate_clean
0,$576.00,576.00,---,NaN
1,$760.00,760.00,---,NaN
2,$91.54,91.54,---,NaN
3,$63.33,63.33,---,NaN
4,$75.93,75.93,---,NaN


In [ ]:
rate_df[[
    "proc_code",
    "procedure_description",
    "basic_rate_clean"
]].isna().mean()


,0
proc_code,0.0
procedure_description,0.0
basic_rate_clean,0.0


In [ ]:


rate_df["proc_code_norm"] = rate_df["proc_code"].str.strip().str.upper()

coverage_rate = rate_df["proc_code_norm"].isin(df["code"]).mean()
coverage_rate


np.float64(0.9014999226843977)

In [ ]:
def lookup(code):
    return rate_df.loc[
        rate_df["proc_code"] == code,
        ["proc_code", "procedure_description", "basic_rate_clean"]
    ]

lookup("00100")
lookup("0001U")


,proc_code,procedure_description,basic_rate_clean
0,0001U,RBC DNA HEA 35 AG 11 BLD GRP,576.0
